# PyAutoGUI 学习笔记（入门速通）

> 目标：掌握安全机制、鼠标控制、键盘模拟、截图与图像识别、消息框交互。

PyAutoGUI 是一个跨平台的 GUI 自动化库，支持 Windows / macOS / Linux。它的核心能力是**用代码模拟人的鼠标和键盘操作**：移动鼠标、点击、拖拽、输入文字、按组合键，还能截图并在屏幕上查找图片位置[citation:4]。

**环境准备**
```bash
pip install pyautogui
```

**依赖说明**：Windows 无需额外依赖；macOS 需要 `pyobjc`；Linux 需要 `python3-xlib`。截图和图像识别功能依赖 Pillow，Linux 上可能还需安装 `scrot`[citation:3][citation:10]。

> ⚠️ **重要提醒**：本 Notebook 中的鼠标键盘操作会真实控制你的电脑。运行代码前请先阅读第 1 节的安全机制，并确保在可控环境中测试。

In [1]:
import pyautogui
import time
print("PyAutoGUI version:", pyautogui.__version__)
print("屏幕分辨率:", pyautogui.size())

PyAutoGUI version: 0.9.54
屏幕分辨率: Size(width=1920, height=1080)


## 1. 安全机制：FAILSAFE 与 PAUSE

自动化脚本最怕“失控”——鼠标乱点、键盘乱敲，而你来不及关掉程序。PyAutoGUI 提供了两道保险。

**① FAILSAFE（故障安全）**
- 默认启用（`True`）。当鼠标移动到屏幕**左上角 (0, 0)** 时，PyAutoGUI 立即抛出 `FailSafeException` 并终止程序[citation:1][citation:7]。
- 这是**救命功能**：脚本失控时，把鼠标甩到左上角就能刹车。

**② PAUSE（全局暂停）**
- `pyautogui.PAUSE = 1` 表示每次 PyAutoGUI 函数调用后自动等待 1 秒[citation:7]。
- 作用：给 GUI 应用留出响应时间，避免操作太快导致界面来不及反应。
- 对于需要精确控制时序的场景，可以在关键操作之间手动 `time.sleep()`。

**建议**：调试期间始终保留 `FAILSAFE = True`，并把 `PAUSE` 设为 0.5~1 秒。确认脚本稳定后，再根据实际需要调整。

In [2]:
# 配置安全机制（强烈建议在脚本开头设置）
pyautogui.FAILSAFE = True   # 鼠标移到左上角紧急停止
pyautogui.PAUSE = 0.5        # 每次操作后暂停 0.5 秒

print("FAILSAFE:", pyautogui.FAILSAFE)
print("PAUSE:", pyautogui.PAUSE)
print("当前鼠标位置:", pyautogui.position())

FAILSAFE: True
PAUSE: 0.5
当前鼠标位置: Point(x=355, y=387)


## 2. 鼠标控制

鼠标操作围绕**坐标**展开。PyAutoGUI 的坐标系：原点 `(0, 0)` 在屏幕**左上角**，X 向右增大，Y 向**下**增大（与数学坐标系 Y 方向相反）[citation:3]。

| 函数 | 作用 | 关键参数 |
|---|---|---|
| `position()` | 获取当前鼠标坐标 | 返回 `(x, y)` |
| `size()` | 获取屏幕分辨率 | 返回 `(width, height)`，仅主显示器 |
| `moveTo(x, y, duration)` | 移动到绝对坐标 | `duration` 为移动耗时（秒），让移动更平滑 |
| `moveRel(xOffset, yOffset)` | 相对当前位置移动 | 偏移量可为负数 |
| `click(x, y, clicks, button)` | 点击 | `button` 可选 `'left'`/`'right'`/`'middle'` |
| `doubleClick()` / `rightClick()` | 双击 / 右键 | 语法糖，等价于设置 `click` 参数 |
| `dragTo(x, y, duration)` | 拖拽到目标坐标 | 需配合 `mouseDown` / `mouseUp`，或直接用 `dragTo` |
| `scroll(amount)` | 滚动滚轮 | 正数向上，负数向下 |

**易错点**
- PyAutoGUI **只可靠支持主显示器**，多显示器环境下鼠标操作可能不生效[citation:3]。
- `duration` 为 0 或不指定时，移动是**瞬移**；设成 0.2~1 秒会更自然，也便于观察。
- macOS Retina 屏和 Windows 缩放可能导致坐标需要换算（如 Retina 上坐标值可能需除以 2）[citation:13]。

In [4]:
# ⚠️ 运行前确保鼠标不在左上角，且你清楚它会移动
print("当前鼠标:", pyautogui.position())

# 移动到屏幕中央附近（需等待，便于观察）
w, h = pyautogui.size()
pyautogui.moveTo(w//2, h//2, duration=1)

# 点击示例（取消注释以执行）
# pyautogui.click(w//2, h//2)
# pyautogui.rightClick()
# pyautogui.doubleClick()

# 滚动示例：向上滚 3 格
# pyautogui.scroll(3)

print("以上操作需取消注释才会执行")

当前鼠标: Point(x=365, y=313)
以上操作需取消注释才会执行


## 3. 键盘控制

键盘操作发送到**当前拥有焦点的窗口**，和真人按键效果相同[citation:3]。

| 函数 | 作用 | 说明 |
|---|---|---|
| `write(text, interval)` | 输入字符串 | `interval` 为每个字符间隔；**对中文支持不好**，建议仅用于英文[citation:1] |
| `press(key)` | 按下并释放单个键 | 键名见下方列表 |
| `keyDown(key)` / `keyUp(key)` | 按住 / 释放 | 用于组合操作，如按住 Shift 再按方向键 |
| `hotkey(*keys)` | 组合键 | 按顺序按下，再按相反顺序释放，如 `hotkey('ctrl', 'c')` |
| `typewrite(list)` | 按列表依次按键 | `['a', 'b', 'left', 'enter']` |

**常用键名**：`'a'`~`'z'`、`'0'`~`'9'`、`'enter'`、`'space'`、`'tab'`、`'esc'`、`'backspace'`、`'up'`/`'down'`/`'left'`/`'right'`、`'f1'`~`'f12'`、`'ctrl'`、`'shift'`、`'alt'`[citation:1]。

**中文输入问题**：`write()` 对中文支持不佳。常见替代方案是用 `pyperclip` 把中文复制到剪贴板，再用 `hotkey('ctrl', 'v')` 粘贴。

**易错点**：按键发送到当前激活窗口。如果脚本运行时你切到了其他窗口，按键会打到错误的程序中。建议在操作前加 `time.sleep(3)`，留出手动切换窗口的时间。

In [ ]:
# ⚠️ 运行前请打开记事本并让它获得焦点
# time.sleep(3)  # 给 3 秒切换到记事本

# 输入英文
# pyautogui.write('Hello PyAutoGUI!', interval=0.1)

# 按键
# pyautogui.press('enter')

# 组合键：全选
# pyautogui.hotkey('ctrl', 'a')

# 按住 Shift 再按方向键（选中文字）
# pyautogui.keyDown('shift')
# pyautogui.press(['left', 'left', 'left'])
# pyautogui.keyUp('shift')

print("以上键盘操作需取消注释并切换窗口后执行")

## 4. 截图与图像识别

这是 PyAutoGUI 最有价值的进阶功能：**不用记坐标**，而是让程序在屏幕上“找图”，找到就点。

| 函数 | 作用 | 返回 |
|---|---|---|
| `screenshot(region=None)` | 截取屏幕 | Pillow `Image` 对象；`region=(left, top, width, height)` 可截局部 |
| `locateOnScreen(image, confidence)` | 在屏幕上查找图片 | 返回 `(left, top, width, height)` 或抛出 `ImageNotFoundException` |
| `locateCenterOnScreen(image)` | 查找并返回中心坐标 | 返回 `(x, y)`，可直接传给 `click()` |
| `center(box)` | 把边界框转为中心点 | 接受 `locateOnScreen` 的返回值 |

**使用流程**：先用截图工具把要点击的按钮/图标保存为 PNG → 代码中用 `locateCenterOnScreen('button.png')` 找到坐标 → `click()`。

**关键参数 `confidence`**：允许一定程度的像素差异（0~1），需要安装 OpenCV（`pip install opencv-python`）。界面缩放、轻微渲染差异时很有用[citation:10]。

**易错点**
- 从 0.9.41 版本起，找不到图片会**抛出异常**而不是返回 `None`，需要用 `try/except` 包裹[citation:10]。
- 图像识别耗时约 1~2 秒（1080p 屏幕），不适合对实时性要求极高的场景[citation:10]。
- 截图文件的分辨率、缩放比例必须与当前屏幕显示一致，否则匹配失败。

In [ ]:
# 截图示例
# im = pyautogui.screenshot()
# im.save('full_screen.png')

# 只截左上角区域（left=0, top=0, width=300, height=200）
# im = pyautogui.screenshot(region=(0, 0, 300, 200))
# im.save('region.png')

print("截图代码已注释，取消注释即可运行")

# 图像识别示例（需要准备一张 button.png）
# try:
#     pos = pyautogui.locateCenterOnScreen('button.png', confidence=0.8)
#     if pos:
#         pyautogui.click(pos)
# except pyautogui.ImageNotFoundException:
#     print("屏幕上未找到图片")

## 5. 消息框

PyAutoGUI 内置了几种简单的弹窗，适合在脚本中暂停等待确认或提示信息。这些函数会**阻塞程序**，直到用户操作。

| 函数 | 行为 | 返回值 |
|---|---|---|
| `alert(text)` | 显示信息 + OK 按钮 | 点击后返回 `'OK'` |
| `confirm(text)` | 显示信息 + OK / Cancel | 返回按钮文本 |
| `prompt(text)` | 输入框 + OK / Cancel | 返回输入的字符串 |
| `password(text, mask)` | 密码输入框 | 返回输入的字符串，`mask` 为掩码字符 |

**用途**：调试时在关键步骤前弹窗确认，避免脚本一运行就失控；或在长时间任务中提示进度。

In [ ]:
# 弹窗示例（会阻塞，手动点击后继续）
# pyautogui.alert('脚本即将开始，请确保目标窗口已激活')
# choice = pyautogui.confirm('是否继续？', buttons=['继续', '取消'])
# name = pyautogui.prompt('请输入你的名字：')

print("弹窗代码已注释，取消注释可体验")

## 6. 综合练习

1. 写一个脚本：打开记事本，输入 “Hello from PyAutoGUI”，然后全选并复制。
2. 结合 `locateCenterOnScreen`：截取计算器的一个数字按钮，让程序自动找到并点击它。
3. 设计一个安全的自动化流程：先 `alert` 确认，执行鼠标移动，再 `confirm` 检查结果，全程保留 `FAILSAFE`。

**参考答案（记事本示例）**

In [ ]:
# 安全机制
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0.5

def notepad_demo():
    # 提示用户准备
    pyautogui.alert('请打开记事本并让它获得焦点，点击 OK 后 3 秒开始')
    time.sleep(3)
    
    # 输入文字
    pyautogui.write('Hello from PyAutoGUI', interval=0.05)
    pyautogui.press('enter')
    pyautogui.write('Automation is fun!', interval=0.05)
    
    # 全选 + 复制
    pyautogui.hotkey('ctrl', 'a')
    pyautogui.hotkey('ctrl', 'c')
    
    print("已输入并复制。请手动粘贴验证。")

# notepad_demo()  # 取消注释并确保记事本在前台后运行

## 常见坑点总结

- **操作的是当前激活窗口**：脚本运行时切窗口，按键会打到别处。用 `time.sleep(3)` 留出切换时间。
- **`FAILSAFE` 是最后防线**：调试时务必保持 `True`，鼠标甩到左上角即可停止。
- **`write()` 不支持中文**：中文输入用 `pyperclip.copy()` + `hotkey('ctrl', 'v')` 绕过。
- **多显示器只支持主屏**：PyAutoGUI 的鼠标操作在主显示器上才可靠[citation:3]。
- **图像识别找不到会抛异常**：`locateOnScreen` 从 0.9.41 起抛 `ImageNotFoundException`，用 `try/except` 处理[citation:10]。
- **`confidence` 需要 OpenCV**：没装 `opencv-python` 时传这个参数会报错。
- **坐标缩放问题**：macOS Retina 或 Windows 显示缩放可能导致坐标偏差，必要时做换算[citation:13]。
- **速度太快界面来不及响应**：用 `PAUSE` 或手动 `time.sleep` 控制节奏。
- **测试环境要安全**：先在记事本、计算器等无害程序中测试，确认逻辑无误再用于真实场景。